# AgentFinder — Laporan Machine Learning
**Topik:** Rekomendasi AI Coding Agents dan Multi-Agent Terminals.  
**Algoritma:** KNN (rekomendasi), Multinomial Naive Bayes dan Decision Tree (klasifikasi).  
**Dataset:** crawling otomatis metadata dan README repository publik GitHub.  

**Penting:** Pengumpulan via GitHub API tidak identik dengan HTML scraping. Konfirmasi kriteria dosen. Kategori awal dibuat dengan *weak labeling* berdasarkan aturan kata kunci, bukan anotasi pakar. Tidak ada dataset atau akurasi buatan.

## 1. Persiapan proyek
Upload **AgentFinder_Project.zip** yang disertakan bersama notebook. Sel pertama akan mengekstrak struktur kode yang sama dengan VS Code. File hasil Colab akan disalin kembali ke folder ini.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()  # pilih AgentFinder_Project.zip
name = next((n for n in uploaded if n.endswith('.zip')), None)
if not name: raise ValueError('Upload AgentFinder_Project.zip')
with zipfile.ZipFile(name) as z: z.extractall('/content')
os.chdir('/content/AgentFinder')
print('Project root:', os.getcwd())

## 2. Instal dependensi
Library digunakan untuk crawling, TF-IDF, KNN, Naive Bayes, Decision Tree, dan evaluasi. Token GitHub bersifat **opsional** namun membantu batas permintaan API.

In [ ]:
%pip -q install -r requirements.txt
from getpass import getpass
import os
answer = input('Gunakan GitHub token opsional? ketik y jika ya: ').strip().lower()
if answer == 'y':
    os.environ['GITHUB_TOKEN'] = getpass('GitHub PAT read-only (tidak ditampilkan): ').strip()
print('Setup selesai. Jangan menampilkan token di output.')

## 3. Crawling dokumen sumber asli
Pencarian repository dilakukan dengan GitHub REST API. Setiap README diambil secara otomatis sebagai dokumen sumber. URL, waktu pengumpulan, dan status unduh disimpan. Program menghapus duplikasi. Jika kuota GitHub tercapai, gunakan token read-only dan jalankan kembali.

In [ ]:
from ml.pipeline import crawl, DATA, MODELS
import os
movies_not_used = True  # proyek ini sepenuhnya tentang developer tools
agents = crawl(token=os.getenv('GITHUB_TOKEN') or None, per_query=20, max_repos=100)
print('Jumlah data valid:', len(agents))
display(agents[['name','category','description','source_url']].head(10))

## 4. Audit dataset
Pemberian kategori dilakukan lewat aturan kata kunci sehingga rawan salah label. Periksa contoh per kategori serta `data/label_audit.csv` dan dokumentasi aslinya. Jika ingin koreksi label, ubah kategori di `data/agents_clean.csv` sebelum training. Jangan menebak data yang tidak tersedia.

In [ ]:
import pandas as pd
raw = pd.read_csv(DATA/'agents_raw.csv')
clean = pd.read_csv(DATA/'agents_clean.csv')
print('Raw:', len(raw), '| Clean:', len(clean))
display(clean['category'].value_counts().rename_axis('category').to_frame('total'))
display(pd.read_csv(DATA/'label_audit.csv').sample(min(12,len(raw)),random_state=42))

## 5. Training dan pembagian data
Holdout split 75% training / 25% testing dengan `stratify`, `random_state=42`. TF-IDF model klasifikasi hanya belajar vocabulary pada training partition. KNN menggunakan dokumen katalog untuk mencari tetangga teks terdekat berdasarkan cosine distance. Program akan berhenti bila data tidak cukup; dataset tidak dipalsukan.

In [ ]:
from ml.pipeline import train
report = train()
print('Jumlah train:', report['train_size'], '| test:',report['test_size'])

## 6. Evaluasi Naive Bayes vs Decision Tree
Bandingkan Accuracy, Precision, Recall, F1 macro dan confusion matrix. Ingat bahwa ini mengukur kemampuan memprediksi **weak labels**, bukan kebenaran kategori yang telah divalidasi pakar.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
for key, label in [('naive_bayes','Naive Bayes'),('decision_tree','Decision Tree')]:
    m=report[key]
    print(label, {x:m[x] for x in ['accuracy','precision_macro','recall_macro','f1_macro']})
    plt.figure(figsize=(5,4))
    plt.imshow(m['confusion_matrix'],cmap='Reds')
    plt.xticks(range(3),report['classes'],rotation=35,ha='right')
    plt.yticks(range(3),report['classes'])
    plt.title('Confusion matrix — '+label)
    plt.xlabel('Predicted');plt.ylabel('True')
    for i in range(3):
        for j in range(3):plt.text(j,i,m['confusion_matrix'][i][j],ha='center',va='center')
    plt.colorbar();plt.tight_layout();plt.show()

## 7. Demo rekomendasi KNN dan klasifikasi
Demonstrasikan satu contoh. Skor similarity menunjukkan kemiripan teks, bukan probabilitas bahwa tool paling baik. Precision@5 memerlukan penilaian relevansi independen dan karena itu tidak dihitung secara palsu.

In [ ]:
import joblib
q='I need a terminal to manage multiple coding agent sessions in parallel'
b=joblib.load(MODELS/'knn.joblib')
d,ids=b['model'].kneighbors(b['vectorizer'].transform([q]),n_neighbors=min(5,len(b['catalog'])))
for distance,i in zip(d[0],ids[0]):
    x=b['catalog'][int(i)]
    if 1-distance>0:print(x['name'],' | ',x['category'],' | similarity',round(1-distance,3),' | ',x['github_url'])
for label in ['naive_bayes','decision_tree']:
    print(label,'→',joblib.load(MODELS/(label+'.joblib')).predict([q])[0])

## 8. Ekspor artefak untuk VS Code
Hasil ZIP berisi `data/` dan `models/` yang harus diekstrak ke dalam folder AgentFinder. **Jangan masukkan token rahasia.** Semua kode aplikasi sudah ada dalam `AgentFinder_Project.zip`.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/AgentFinder_Artifacts','zip',root_dir='/content/AgentFinder',base_dir='data')
# Add models to the existing archive
with zipfile.ZipFile('/content/AgentFinder_Artifacts.zip','a',zipfile.ZIP_DEFLATED) as archive:
    for p in MODELS.glob('*'):
        if p.is_file():archive.write(p,'models/'+p.name)
files.download('/content/AgentFinder_Artifacts.zip')
print('Ekstrak ZIP ke root AgentFinder di VS Code: data/ dan models/.')

## 9. Ringkasan dan kesimpulan untuk presentasi
Isi bagian ini setelah menjalankan semua kode dengan dataset nyata:
- Jumlah data awal, bersih, dan distribusi kelas.
- Skor aktual Naive Bayes dan Decision Tree beserta penjelasan confusion matrix.
- Lima hasil rekomendasi KNN untuk contoh kebutuhan dan tautan sumbernya.
- Kelemahan: pelabelan berbasis aturan, kelengkapan README, bias popularitas repository dan perubahan katalog dari waktu ke waktu.
- Usulan berikutnya: audit anotasi manual dan relevansi rekomendasi berbasis penilaian manusia.